# STL Decomposition

**STL** stands for **Seasonal and Trend decomposition using LOESS** (LOcally
Estimated Scatterplot Smoothing). It was developed by Cleveland, Cleveland,
McRae, and Terpenning (1990) and addresses all the major limitations of
classical decomposition.

LOESS is a **non-parametric regression** method that fits simple models to
localized subsets of data, producing a smooth curve without assuming a global
functional form. STL applies LOESS iteratively to separate trend and seasonal
components.

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL, seasonal_decompose

# Plotting defaults
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

In [ ]:
# --- Airline Passengers (monthly, 1949-1960) ---
airline = pd.read_csv(
    DATA_DIR / "airline_passengers.csv",
    index_col="Month",
    parse_dates=True,
)
airline.index.freq = "MS"
airline.columns = ["Passengers"]

passengers = airline["Passengers"]

print("Airline shape:", airline.shape)
airline.head()

## Advantages over Classical Decomposition

| Feature | Classical | STL |
|---|---|---|
| Seasonal component | Fixed (identical every year) | Can change over time |
| Trend smoothness | Fixed by $m$ | User-controlled via `trend` parameter |
| Robustness to outliers | None | Built-in via `robust=True` |
| Edge values | Missing first/last $m/2$ | Estimated (no missing values) |
| Seasonality type | Any period | Any period |
| Model type | Additive or multiplicative | Additive only (but see log trick) |

## Basic STL in statsmodels

In [ ]:
from statsmodels.tsa.seasonal import STL

stl = STL(passengers, period=12, seasonal=13, trend=None, robust=True)
result = stl.fit()
result.plot()
plt.suptitle("STL Decomposition — Airline Passengers", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Accessing individual components
print("Component shapes:")
print(f"  trend    : {result.trend.shape}")
print(f"  seasonal : {result.seasonal.shape}")
print(f"  resid    : {result.resid.shape}")

print(f"\nNaN count in trend:  {result.trend.isna().sum()}")
print(f"NaN count in resid:  {result.resid.isna().sum()}")
print("(No missing values — unlike classical decomposition!)")

In [ ]:
# Verify: observed = trend + seasonal + resid
reconstructed = result.trend + result.seasonal + result.resid
diff = (passengers - reconstructed).abs().max()
print(f"Max reconstruction error: {diff:.2e}")

## Key Parameters

STL's power comes from its tunable parameters. The three most important are:

- **`seasonal`**: Controls how quickly the seasonal component can change.
  Must be odd. Larger values produce a more stable (slowly changing) seasonal
  pattern. Setting `seasonal=99` or higher approximates a fixed seasonal
  pattern (like classical decomposition).

- **`trend`**: Controls the smoothness of the trend-cycle. Must be odd.
  Larger values produce a smoother trend. If `None`, statsmodels chooses a
  sensible default.

- **`robust`**: When `True`, the LOESS fitting uses a robust weighting scheme
  that down-weights outliers, so they affect only the remainder (not the
  trend or seasonal components).

### Effect of the `seasonal` Parameter

In [ ]:
seasonal_values = [7, 13, 31, 99]

fig, axes = plt.subplots(len(seasonal_values), 1, figsize=(12, 12), sharex=True)

for ax, s in zip(axes, seasonal_values):
    res = STL(passengers, period=12, seasonal=s).fit()
    ax.plot(res.seasonal, label=f"seasonal={s}")
    ax.set_ylabel("Seasonal")
    ax.legend(loc="upper right")

axes[0].set_title("Effect of seasonal Parameter on the Seasonal Component")
axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.show()

print("Smaller seasonal → seasonal pattern changes more rapidly.")
print("Larger seasonal → seasonal pattern is nearly fixed (like classical).")

### Effect of the `trend` Parameter

In [ ]:
trend_values = [13, 25, 51, 101]

fig, axes = plt.subplots(len(trend_values), 1, figsize=(12, 12), sharex=True)

for ax, t in zip(axes, trend_values):
    res = STL(passengers, period=12, seasonal=13, trend=t).fit()
    ax.plot(passengers, alpha=0.3, label="Observed")
    ax.plot(res.trend, label=f"trend={t}", linewidth=2, color="tab:red")
    ax.set_ylabel("Trend")
    ax.legend(loc="upper left")

axes[0].set_title("Effect of trend Parameter on Trend Smoothness")
axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.show()

print("Smaller trend → trend follows the data more closely (may capture noise).")
print("Larger trend → smoother trend that only captures long-term direction.")

### Effect of `robust` on Outliers

In [ ]:
# Create data with an artificial outlier
passengers_outlier = passengers.copy()
outlier_idx = "1958-06-01"
passengers_outlier.loc[outlier_idx] = passengers_outlier.loc[outlier_idx] + 250

print(f"Injected outlier at {outlier_idx}:")
print(f"  Original value: {passengers.loc[outlier_idx]}")
print(f"  Outlier value:  {passengers_outlier.loc[outlier_idx]}")

In [ ]:
# Compare robust=False vs robust=True on the outlier data
res_not_robust = STL(passengers_outlier, period=12, seasonal=13, robust=False).fit()
res_robust = STL(passengers_outlier, period=12, seasonal=13, robust=True).fit()

fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)

# robust=False (left)
axes[0, 0].plot(passengers_outlier, alpha=0.5)
axes[0, 0].plot(res_not_robust.trend, color="tab:red", linewidth=2)
axes[0, 0].set_title("robust=False")
axes[0, 0].set_ylabel("Trend")

axes[1, 0].plot(res_not_robust.seasonal, color="tab:purple")
axes[1, 0].set_ylabel("Seasonal")

axes[2, 0].plot(res_not_robust.resid, color="tab:green")
axes[2, 0].axhline(0, color="grey", linestyle="--", linewidth=0.8)
axes[2, 0].set_ylabel("Remainder")

# robust=True (right)
axes[0, 1].plot(passengers_outlier, alpha=0.5)
axes[0, 1].plot(res_robust.trend, color="tab:red", linewidth=2)
axes[0, 1].set_title("robust=True")

axes[1, 1].plot(res_robust.seasonal, color="tab:purple")

axes[2, 1].plot(res_robust.resid, color="tab:green")
axes[2, 1].axhline(0, color="grey", linestyle="--", linewidth=0.8)

plt.suptitle("STL with Artificial Outlier — robust=False vs robust=True", fontsize=14)
plt.tight_layout()
plt.show()

print("With robust=True, the outlier is absorbed into the remainder component,")
print("leaving the trend and seasonal components clean.")

## STL vs Classical Decomposition

Let's put them side by side on the airline passengers data to highlight
the differences.

In [ ]:
# Classical decomposition
classical = seasonal_decompose(passengers, model="additive", period=12)

# STL decomposition
stl_result = STL(passengers, period=12, seasonal=13, robust=True).fit()

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 12), sharex=True)

# Classical (left)
axes[0, 0].plot(classical.observed)
axes[0, 0].set_ylabel("Observed")
axes[0, 0].set_title("Classical Decomposition")

axes[1, 0].plot(classical.trend, color="tab:red")
axes[1, 0].set_ylabel("Trend")

axes[2, 0].plot(classical.seasonal, color="tab:purple")
axes[2, 0].set_ylabel("Seasonal")

axes[3, 0].plot(classical.resid, color="tab:green")
axes[3, 0].axhline(0, color="grey", linestyle="--", linewidth=0.8)
axes[3, 0].set_ylabel("Remainder")

# STL (right)
axes[0, 1].plot(stl_result.observed)
axes[0, 1].set_title("STL Decomposition")

axes[1, 1].plot(stl_result.trend, color="tab:red")

axes[2, 1].plot(stl_result.seasonal, color="tab:purple")

axes[3, 1].plot(stl_result.resid, color="tab:green")
axes[3, 1].axhline(0, color="grey", linestyle="--", linewidth=0.8)

plt.suptitle("Classical vs STL — Airline Passengers (Additive)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Key difference 1: STL has no missing values at edges
print("Missing values comparison:")
print(f"  Classical trend NaN: {classical.trend.isna().sum()}")
print(f"  STL trend NaN:      {stl_result.trend.isna().sum()}")
print(f"  Classical resid NaN: {classical.resid.isna().sum()}")
print(f"  STL resid NaN:      {stl_result.resid.isna().sum()}")

In [ ]:
# Key difference 2: STL seasonal component can change over time
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(classical.seasonal, color="tab:purple")
axes[0].set_title("Classical: Fixed Seasonal Pattern")
axes[0].set_ylabel("Seasonal")

axes[1].plot(stl_result.seasonal, color="tab:purple")
axes[1].set_title("STL: Evolving Seasonal Pattern")
axes[1].set_ylabel("Seasonal")

plt.tight_layout()
plt.show()

# Show the seasonal amplitude changes over time in STL
stl_seasonal_amp = stl_result.seasonal.groupby(stl_result.seasonal.index.year).apply(
    lambda x: x.max() - x.min()
)
print("\nSTL seasonal amplitude by year:")
print(stl_seasonal_amp.round(1))

## Trend and Seasonal Strength

We can quantify how much of the variation in a series is explained by its
trend and seasonal components using **strength measures** from Wang, Smith, &
Hyndman (2006):

**Trend strength:**

$$F_T = 1 - \frac{\text{Var}(R_t)}{\text{Var}(T_t + R_t)}$$

**Seasonal strength:**

$$F_S = 1 - \frac{\text{Var}(R_t)}{\text{Var}(S_t + R_t)}$$

Both measures range from 0 (no trend/seasonality) to 1 (strong
trend/seasonality). Values near 0 mean the component contributes little;
values near 1 mean it dominates.

In [ ]:
def compute_strength(result):
    """Compute trend and seasonal strength from an STL result."""
    R = result.resid
    T = result.trend
    S = result.seasonal

    # Trend strength
    F_T = max(0, 1 - R.var() / (T + R).var())

    # Seasonal strength
    F_S = max(0, 1 - R.var() / (S + R).var())

    return F_T, F_S


F_T, F_S = compute_strength(stl_result)

print("Airline Passengers — STL Strength Measures")
print(f"  Trend strength:    F_T = {F_T:.4f}")
print(f"  Seasonal strength: F_S = {F_S:.4f}")
print(f"\nBoth are high, confirming strong trend and strong seasonality.")

In [ ]:
# Compare with a series that has weak seasonality: random walk
np.random.seed(42)
random_walk = pd.Series(
    np.cumsum(np.random.randn(144)),
    index=passengers.index,
    name="RandomWalk",
)

rw_result = STL(random_walk, period=12, seasonal=13).fit()
F_T_rw, F_S_rw = compute_strength(rw_result)

print("Random Walk — STL Strength Measures")
print(f"  Trend strength:    F_T = {F_T_rw:.4f}")
print(f"  Seasonal strength: F_S = {F_S_rw:.4f}")
print(f"\nAs expected: trend may appear moderate (random walks drift), but")
print(f"seasonal strength is low since there is no real seasonality.")

## Multiplicative Decomposition with STL

STL is inherently **additive only**:

$$y_t = T_t + S_t + R_t$$

But we can handle multiplicative seasonality with a simple trick: apply STL
to the **log-transformed** data, then exponentiate the components.

If $y_t = T_t \times S_t \times R_t$, then:

$$\log(y_t) = \log(T_t) + \log(S_t) + \log(R_t)$$

This is an additive model in the log domain.

In [ ]:
# Step 1: Log-transform
log_passengers = np.log(passengers)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(passengers)
axes[0].set_title("Original Scale (multiplicative seasonality)")
axes[0].set_ylabel("Passengers")

axes[1].plot(log_passengers, color="tab:orange")
axes[1].set_title("Log Scale (now additive seasonality)")
axes[1].set_ylabel("log(Passengers)")

plt.tight_layout()
plt.show()

In [ ]:
# Step 2: Apply STL to log-transformed data
stl_log = STL(log_passengers, period=12, seasonal=13, robust=True)
result_log = stl_log.fit()
result_log.plot()
plt.suptitle("STL on log(Passengers) — Additive in Log Domain", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Step 3: Exponentiate to get multiplicative components in original scale
trend_mult = np.exp(result_log.trend)
seasonal_mult = np.exp(result_log.seasonal)
resid_mult = np.exp(result_log.resid)

# Verify: T * S * R should reconstruct the original series
reconstructed = trend_mult * seasonal_mult * resid_mult
max_error = (passengers - reconstructed).abs().max()
print(f"Max reconstruction error: {max_error:.2e}")

In [ ]:
# Plot multiplicative components
fig, axes = plt.subplots(4, 1, figsize=(12, 12), sharex=True)

axes[0].plot(passengers)
axes[0].set_ylabel("Observed")
axes[0].set_title("STL Multiplicative Decomposition (via log transform)")

axes[1].plot(trend_mult, color="tab:red")
axes[1].set_ylabel("Trend")

axes[2].plot(seasonal_mult, color="tab:purple")
axes[2].axhline(1, color="grey", linestyle="--", linewidth=0.8)
axes[2].set_ylabel("Seasonal\n(multiplicative)")

axes[3].plot(resid_mult, color="tab:green")
axes[3].axhline(1, color="grey", linestyle="--", linewidth=0.8)
axes[3].set_ylabel("Remainder\n(multiplicative)")
axes[3].set_xlabel("Date")

plt.tight_layout()
plt.show()

print("The multiplicative seasonal component oscillates around 1.0,")
print("and the remainder is more uniform than the additive version.")

In [ ]:
# Compare residuals: additive STL vs multiplicative STL (via log)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(stl_result.resid, color="tab:green", alpha=0.7)
axes[0].axhline(0, color="grey", linestyle="--", linewidth=0.8)
axes[0].set_title("Additive STL Residuals")
axes[0].set_ylabel("Residual")

axes[1].plot(resid_mult, color="tab:orange", alpha=0.7)
axes[1].axhline(1, color="grey", linestyle="--", linewidth=0.8)
axes[1].set_title("Multiplicative STL Residuals (via log)")
axes[1].set_ylabel("Residual")

plt.suptitle("Additive vs Multiplicative STL Residuals", fontsize=13)
plt.tight_layout()
plt.show()

print("The additive residuals show growing variance (heteroscedasticity).")
print("The multiplicative residuals are much more uniform — a better fit.")

## STL for Forecasting (Preview)

Decomposition is not just for understanding — it can also be used as part of
a forecasting pipeline. The idea is straightforward:

1. **Decompose** the series using STL
2. **Forecast** each component separately (or at least the seasonally adjusted
   series $T_t + R_t$)
3. **Recombine** the forecasts

This is sometimes called **STL + ETS** or **STL + ARIMA** forecasting.
We will cover this in detail in Chapters 06 and 07.

In [ ]:
# Quick demo: separate the seasonally adjusted series
seasonally_adjusted = passengers - stl_result.seasonal

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(passengers, label="Original", alpha=0.7)
axes[0].plot(seasonally_adjusted, label="Seasonally Adjusted", linewidth=2, color="tab:red")
axes[0].set_title("Seasonally Adjusted Series = Original − Seasonal Component")
axes[0].set_ylabel("Passengers")
axes[0].legend()

axes[1].plot(stl_result.seasonal, color="tab:purple")
axes[1].set_title("Seasonal Component (to be re-added to the forecast)")
axes[1].set_ylabel("Seasonal")
axes[1].set_xlabel("Date")

plt.tight_layout()
plt.show()

print("Forecasting approach:")
print("  1. Forecast the seasonally adjusted series (Trend + Remainder)")
print("  2. Extrapolate the seasonal pattern into the forecast horizon")
print("  3. Add them together: forecast = forecast_adjusted + seasonal")
print("\nMore on this in Chapter 06 (ETS) and Chapter 07 (ARIMA).")

## Summary

**STL (Seasonal and Trend decomposition using LOESS)** is the recommended
general-purpose decomposition method for most time series work.

### Key Takeaways

| Parameter | Effect | Guidance |
|---|---|---|
| `seasonal` | Rate of change of seasonal component | Odd, $\geq 7$. Larger = more stable. |
| `trend` | Smoothness of trend | Odd. Larger = smoother. `None` for auto. |
| `robust` | Outlier handling | `True` for real-world data with outliers. |

### Strength Measures

$$F_T = 1 - \frac{\text{Var}(R_t)}{\text{Var}(T_t + R_t)} \qquad F_S = 1 - \frac{\text{Var}(R_t)}{\text{Var}(S_t + R_t)}$$

### Multiplicative Trick

STL is additive only, but applying it to $\log(y_t)$ and exponentiating the
components gives a multiplicative decomposition.

### What's Next

- Chapter 05: Autocorrelation, ACF, and PACF
- Chapter 06-07: Using decomposition as part of forecasting pipelines